# 07a · Selección de checkpoint y escala de guía — Difusión (validación)

**Fase CRISP-DM:** 5. Evaluación (solo selección, con **validación**; test no se usa).

Mismo protocolo que la VAE (notebook 06): embeddings **DINOv2-ViT-L/14** de las 510 imágenes reales de validación a 128 px; métricas **KD** (criterio), FD, precisión/recall y densidad/cobertura.

1. **Checkpoint:** se evalúan los pesos EMA desde `MIN_STEP` (los anteriores aún arrastran pesos iniciales de la EMA) con guía w = 2.
2. **Escala de guía (classifier-free guidance, Ho y Salimans, 2022):** con el mejor checkpoint se barre w ∈ {1, 1.5, 2, 3, 4}.
   Es el compromiso fidelidad–diversidad que tu estado del arte documenta (Dhariwal y Nichol, 2021; Ho y Salimans, 2022; Sariyildiz et al., 2023): se espera que al subir w suba la precisión y baje el recall.
3. Se elige el par (checkpoint, w) con **menor KD**.

**Muestras por evaluación:** 510 (30 por clase), por costo de muestreo (50 pasos DDIM × 2 evaluaciones por paso). KD es insesgada con muestras pequeñas; FD solo se compara dentro de esta tabla.
Es **reanudable**: los resultados se guardan fila a fila en Drive.

**Entorno:** GPU **L4**, versión **2026.07**. Duración ≈ 1–1.5 h.

## Paso 1 · Entorno y parámetros

In [1]:
import sys, os, json, shutil, hashlib, zipfile, time, io, random, re, subprocess, copy, math
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import scipy.linalg
import matplotlib.pyplot as plt
from PIL import Image
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python:", sys.version.split()[0], "| PyTorch:", torch.__version__,
      "| GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "⚠️ SIN GPU")
SEED = 42
RES = 128
K_PR = 5
FEATURES = os.environ.get("FEATURES", "dinov2")      # 'mock' solo para pruebas sin GPU
AMP = dict(device_type=DEVICE.type, dtype=torch.bfloat16)

Python: 3.12.13 | PyTorch: 2.11.0+cu128 | GPU: NVIDIA L4


In [2]:
%pip install -q diffusers

In [3]:
MIN_STEP = int(os.environ.get("MIN_STEP", 40_000))
N_POR_CLASE = int(os.environ.get("N_POR_CLASE", 30))
DDIM_STEPS = int(os.environ.get("DDIM_STEPS", 50))
GUIAS = [float(x) for x in os.environ.get("GUIAS", "1,1.5,2,3,4").split(",")]
GUIA_BASE = 2.0
BS_GEN = int(os.environ.get("BS_GEN", 128))

## Paso 2 · Drive y datos

In [4]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/tesis_retail")
    LOCAL_DIR = Path("/content/data")
else:
    PROJECT_DIR = Path(os.environ.get("PROJECT_DIR", "./tesis_retail")).resolve()
    LOCAL_DIR = Path(os.environ.get("LOCAL_DIR", "./data_local")).resolve()
LOGS = PROJECT_DIR / "logs"
SEL_DIR = PROJECT_DIR / "seleccion"
CKPT = PROJECT_DIR / "checkpoints"
for d in [SEL_DIR, LOCAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

LOG03 = json.loads((LOGS / "03_piloto.json").read_text())

def cargar_zip(nombre, sha):
    z = LOCAL_DIR / nombre
    if not z.exists():
        shutil.copy2(PROJECT_DIR / "datasets" / nombre, z)
    assert sha256_file(z) == sha, f"⚠️ {nombre} no coincide con el del notebook 03"
    with zipfile.ZipFile(z) as zf:
        lab = dict(json.loads(zf.read("dataset.json"))["labels"])
        nombres = sorted(lab)
        X = np.stack([np.asarray(Image.open(io.BytesIO(zf.read(n))).convert("RGB")) for n in nombres])
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.tensor([lab[n] for n in nombres])

X_train, Y_train = cargar_zip(LOG03["datasets"]["train_zip"], LOG03["datasets"]["train_sha256"])
X_val, Y_val = cargar_zip(LOG03["datasets"]["val_zip"], LOG03["datasets"]["val_sha256"])
N_CLASSES = int(Y_train.max()) + 1
CLASS_MAP = json.loads((PROJECT_DIR / "manifest" / "class_map.json").read_text())
CLASES = [c for c, _ in sorted(CLASS_MAP.items(), key=lambda kv: kv[1])]
NOMBRE_CORTO = [c.split("-")[1] if c.count("-") >= 2 else c for c in CLASES]
print(f"train {tuple(X_train.shape)} | val {tuple(X_val.shape)} | clases {N_CLASSES} ✔")

Mounted at /content/drive
train (3060, 3, 128, 128) | val (510, 3, 128, 128) | clases 17 ✔


## Paso 3 · Extractor DINOv2

In [5]:
import torchvision.transforms.functional as TF
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
if FEATURES == "dinov2":
    dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vitl14").eval().to(DEVICE)
else:
    torch.manual_seed(0)
    dino = nn.Sequential(nn.AdaptiveAvgPool2d(8), nn.Flatten(), nn.Linear(3 * 64, 64)).eval().to(DEVICE)

@torch.no_grad()
def embeddings(x_m11, bs=128):
    # x en [-1, 1], (N,3,128,128) → (N, D) float64. Mismo preprocesamiento que los notebooks 02 y 06.
    out = []
    for i in range(0, len(x_m11), bs):
        x = ((x_m11[i:i + bs].float().cpu() + 1) / 2).clamp(0, 1)
        x = TF.resize(x, [224, 224], interpolation=TF.InterpolationMode.BICUBIC, antialias=True)
        out.append(dino(((x - MEAN) / STD).to(DEVICE)).float().cpu())
    return torch.cat(out).double().numpy()

def a_m11(x_uint8):
    return x_uint8.float() / 127.5 - 1.0

F_VAL = embeddings(a_m11(X_val))
print("Embeddings de validación:", F_VAL.shape)

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitl14/dinov2_vitl14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitl14_pretrain.pth


100%|██████████| 1.13G/1.13G [00:03<00:00, 378MB/s]


Embeddings de validación: (510, 1024)


## Paso 4 · Métricas

In [6]:
def fd(a, b):
    mu1, mu2 = a.mean(0), b.mean(0)
    s1, s2 = np.cov(a, rowvar=False), np.cov(b, rowvar=False)
    cm = scipy.linalg.sqrtm(s1 @ s2)
    cm = cm[0] if isinstance(cm, tuple) else cm
    if not np.isfinite(cm).all():
        off = np.eye(len(s1)) * 1e-6
        cm = scipy.linalg.sqrtm((s1 + off) @ (s2 + off)); cm = cm[0] if isinstance(cm, tuple) else cm
    return float(((mu1 - mu2) ** 2).sum() + np.trace(s1 + s2 - 2 * cm.real))

def kd(a, b):
    # MMD² insesgado con kernel polinómico (x·y/d + 1)^3 (como KID; Bińkowski et al., 2018)
    d = a.shape[1]
    k = lambda x, y: (x @ y.T / d + 1) ** 3
    kaa, kbb, kab = k(a, a), k(b, b), k(a, b)
    m, n = len(a), len(b)
    return float((kaa.sum() - np.trace(kaa)) / (m * (m - 1)) + (kbb.sum() - np.trace(kbb)) / (n * (n - 1)) - 2 * kab.mean())

def dist(a, b):
    return np.sqrt(np.clip((a ** 2).sum(1)[:, None] + (b ** 2).sum(1)[None, :] - 2 * a @ b.T, 0, None))

def prdc(real, fake, k=K_PR):
    # Precisión/recall (Kynkäänniemi et al., 2019) y densidad/cobertura (Naeem et al., 2020)
    r_rad = np.sort(dist(real, real), 1)[:, k]
    f_rad = np.sort(dist(fake, fake), 1)[:, k]
    d_rf = dist(real, fake)                                   # (n_real, n_fake)
    precision = float((d_rf <= r_rad[:, None]).any(0).mean())
    recall = float((d_rf <= f_rad[None, :]).any(1).mean())
    density = float((d_rf <= r_rad[:, None]).sum(0).mean() / k)
    coverage = float((d_rf.min(1) <= r_rad).mean())
    return {"precision": precision, "recall": recall, "density": density, "coverage": coverage}

def evaluar(real, fake, con_fd=True):
    out = {"KD": kd(real, fake), **prdc(real, fake)}
    if con_fd:
        out["FD"] = fd(real, fake)
    return out

## Paso 5 · Modelo de difusión y muestreo DDIM con guía

In [7]:
import diffusers
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler
RUN_DIR = CKPT / "ddpm_cond128_seed42"
EMAS = sorted((RUN_DIR / "ema").glob("ema_*.pt"))
assert EMAS, "No hay snapshots EMA: ¿terminó el notebook 05?"
sched = DDPMScheduler(num_train_timesteps=1000, beta_schedule="squaredcos_cap_v2")

def cargar_unet(ruta):
    ck = torch.load(ruta, map_location="cpu", weights_only=False)
    cfg = ck["config"]
    m = UNet2DModel(sample_size=cfg["res"], in_channels=3, out_channels=3, layers_per_block=2,
                    block_out_channels=tuple(cfg["channels"]),
                    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D", "AttnDownBlock2D", "AttnDownBlock2D"),
                    up_block_types=("AttnUpBlock2D", "AttnUpBlock2D", "UpBlock2D", "UpBlock2D", "UpBlock2D"),
                    num_class_embeds=cfg["n_classes"] + 1)
    m.load_state_dict(ck["ema"]); return m.eval().to(DEVICE), ck["paso"], cfg["n_classes"]

@torch.no_grad()
def generar_difusion(modelo, null_idx, y, guia, pasos=DDIM_STEPS, seed=SEED, bs=BS_GEN):
    out = []
    for i in range(0, len(y), bs):
        yy = y[i:i + bs].to(DEVICE)
        ddim = DDIMScheduler.from_config(sched.config); ddim.set_timesteps(pasos)
        g = torch.Generator(device=DEVICE).manual_seed(seed + i)
        x = torch.randn(len(yy), 3, RES, RES, device=DEVICE, generator=g)
        y_null = torch.full_like(yy, null_idx)
        for t in ddim.timesteps:
            with torch.autocast(**AMP):
                e = modelo(torch.cat([x, x]), t, class_labels=torch.cat([yy, y_null])).sample.float()
            e_c, e_u = e.chunk(2)
            x = ddim.step(e_u + guia * (e_c - e_u), t, x).prev_sample
        out.append(x.clamp(-1, 1).cpu())
    return torch.cat(out)

Y_SEL = torch.arange(N_CLASSES).repeat_interleave(N_POR_CLASE)
print("diffusers:", diffusers.__version__, "| snapshots EMA:", [e.name for e in EMAS])

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


diffusers: 0.39.0 | snapshots EMA: ['ema_0010000.pt', 'ema_0020000.pt', 'ema_0030000.pt', 'ema_0040000.pt', 'ema_0050000.pt', 'ema_0060000.pt', 'ema_0070000.pt', 'ema_0080000.pt', 'ema_0090000.pt', 'ema_0100000.pt']


## Paso 6 · Selección del checkpoint (w = 2)

In [8]:
CSV_CK = SEL_DIR / "seleccion_difusion_ckpt.csv"
hechos = pd.read_csv(CSV_CK) if CSV_CK.exists() else pd.DataFrame()
filas = hechos.to_dict("records")
for ruta in EMAS:
    paso = int(re.search(r"ema_(\d+)\.pt", ruta.name).group(1))
    if paso < MIN_STEP or (len(hechos) and paso in set(hechos["paso"])):
        continue
    modelo, _, nc = cargar_unet(ruta)
    t0 = time.time(); x = generar_difusion(modelo, nc, Y_SEL, GUIA_BASE); dt = time.time() - t0
    res = evaluar(F_VAL, embeddings(x))
    fila = {"checkpoint": ruta.name, "paso": paso, "guia": GUIA_BASE, **res, "seg_por_img": dt / len(Y_SEL)}
    filas.append(fila); pd.DataFrame(filas).to_csv(CSV_CK, index=False)
    print(f"{ruta.name}: KD {res['KD']:.4f} | FD {res['FD']:.1f} | P {res['precision']:.3f} | R {res['recall']:.3f} | "
          f"{dt/len(Y_SEL):.2f} s/img", flush=True)
    del modelo; torch.cuda.empty_cache() if DEVICE.type == "cuda" else None
SEL_CK = pd.DataFrame(filas).sort_values("paso").reset_index(drop=True)
MEJOR_CK = SEL_CK.loc[SEL_CK["KD"].idxmin()]
print("\n★ Mejor checkpoint (w=2):", MEJOR_CK["checkpoint"])
SEL_CK

ema_0040000.pt: KD 4.5756 | FD 1628.4 | P 0.053 | R 0.202 | 0.76 s/img
ema_0050000.pt: KD 3.1941 | FD 1278.6 | P 0.175 | R 0.447 | 0.76 s/img
ema_0060000.pt: KD 2.3234 | FD 1048.5 | P 0.247 | R 0.678 | 0.76 s/img
ema_0070000.pt: KD 1.9786 | FD 945.1 | P 0.314 | R 0.629 | 0.76 s/img
ema_0080000.pt: KD 1.7803 | FD 895.3 | P 0.310 | R 0.667 | 0.76 s/img
ema_0090000.pt: KD 1.6889 | FD 870.9 | P 0.324 | R 0.671 | 0.76 s/img
ema_0100000.pt: KD 1.6185 | FD 849.8 | P 0.341 | R 0.694 | 0.76 s/img

★ Mejor checkpoint (w=2): ema_0100000.pt


,checkpoint,paso,guia,KD,precision,recall,density,coverage,FD,seg_por_img
0,ema_0040000.pt,40000,2.0,4.575599,0.052941,0.201961,0.012157,0.031373,1628.434408,0.763893
1,ema_0050000.pt,50000,2.0,3.194100,0.174510,0.447059,0.054902,0.109804,1278.592022,0.762597
2,ema_0060000.pt,60000,2.0,2.323385,0.247059,0.678431,0.107843,0.213725,1048.533924,0.762625
3,ema_0070000.pt,70000,2.0,1.978603,0.313725,0.629412,0.136863,0.272549,945.106315,0.762782
4,ema_0080000.pt,80000,2.0,1.780258,0.309804,0.666667,0.156471,0.303922,895.272498,0.762909
5,ema_0090000.pt,90000,2.0,1.688948,0.323529,0.670588,0.165490,0.327451,870.870244,0.762899
6,ema_0100000.pt,100000,2.0,1.618467,0.341176,0.694118,0.177647,0.347059,849.816061,0.762900


## Paso 7 · Barrido de la escala de guía w (mejor checkpoint)

In [9]:
CSV_W = SEL_DIR / "seleccion_difusion_guia.csv"
hechos = pd.read_csv(CSV_W) if CSV_W.exists() else pd.DataFrame()
filas = [r for r in hechos.to_dict("records") if r["checkpoint"] == MEJOR_CK["checkpoint"]]
ya = {r["guia"] for r in filas}
if GUIA_BASE not in ya:                                    # reutiliza la evaluación del Paso 6
    filas.append({k: MEJOR_CK[k] for k in ["checkpoint", "paso", "guia", "KD", "FD", "precision", "recall", "density", "coverage", "seg_por_img"]})
modelo, _, nc = cargar_unet(RUN_DIR / "ema" / MEJOR_CK["checkpoint"])
for w in GUIAS:
    if w in {r["guia"] for r in filas}:
        continue
    t0 = time.time(); x = generar_difusion(modelo, nc, Y_SEL, w); dt = time.time() - t0
    res = evaluar(F_VAL, embeddings(x))
    filas.append({"checkpoint": MEJOR_CK["checkpoint"], "paso": int(MEJOR_CK["paso"]), "guia": w, **res, "seg_por_img": dt / len(Y_SEL)})
    pd.DataFrame(filas).to_csv(CSV_W, index=False)
    print(f"w={w}: KD {res['KD']:.4f} | FD {res['FD']:.1f} | P {res['precision']:.3f} | R {res['recall']:.3f}", flush=True)
pd.DataFrame(filas).to_csv(CSV_W, index=False)
SEL_W = pd.DataFrame(filas).sort_values("guia").reset_index(drop=True)
MEJOR = SEL_W.loc[SEL_W["KD"].idxmin()]
print(f"\n★ Elegido: {MEJOR['checkpoint']} con w = {MEJOR['guia']}")
SEL_W

w=1.0: KD 2.3774 | FD 1141.2 | P 0.212 | R 0.651
w=1.5: KD 1.8312 | FD 935.7 | P 0.298 | R 0.651
w=3.0: KD 1.6939 | FD 862.0 | P 0.357 | R 0.625
w=4.0: KD 2.9230 | FD 1250.8 | P 0.208 | R 0.598

★ Elegido: ema_0100000.pt con w = 2.0


,checkpoint,paso,guia,KD,FD,precision,recall,density,coverage,seg_por_img
0,ema_0100000.pt,100000,1.0,2.377431,1141.245236,0.211765,0.650980,0.101961,0.229412,0.762795
1,ema_0100000.pt,100000,1.5,1.831214,935.657011,0.298039,0.650980,0.147843,0.300000,0.762791
2,ema_0100000.pt,100000,2.0,1.618467,849.816061,0.341176,0.694118,0.177647,0.347059,0.762900
3,ema_0100000.pt,100000,3.0,1.693896,861.972220,0.356863,0.625490,0.183529,0.350980,0.762776
4,ema_0100000.pt,100000,4.0,2.922963,1250.766250,0.207843,0.598039,0.090588,0.194118,0.762846


## Paso 8 · Gráficos y muestras del elegido

In [10]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(SEL_CK["paso"] / 1000, SEL_CK["KD"], "o-"); ax[0].set_title("KD vs paso (w=2) ↓"); ax[0].set_xlabel("miles de pasos")
ax[0].axvline(MEJOR_CK["paso"] / 1000, c="red", alpha=.4)
ax[1].plot(SEL_W["guia"], SEL_W["KD"], "o-"); ax[1].set_title("KD vs escala de guía ↓"); ax[1].set_xlabel("w")
ax[1].axvline(MEJOR["guia"], c="red", alpha=.4)
ax[2].plot(SEL_W["guia"], SEL_W["precision"], "o-", label="precisión")
ax[2].plot(SEL_W["guia"], SEL_W["recall"], "s-", label="recall")
ax[2].plot(SEL_W["guia"], SEL_W["coverage"], "^-", label="cobertura")
ax[2].set_ylim(0, 1); ax[2].legend(); ax[2].set_title("Compromiso fidelidad–diversidad vs w"); ax[2].set_xlabel("w")
fig.tight_layout(); fig.savefig(SEL_DIR / "seleccion_difusion.png", dpi=150); plt.show()

import torchvision.utils as vutils
x = generar_difusion(modelo, nc, torch.arange(N_CLASSES).repeat(6), float(MEJOR["guia"]), seed=123)
vutils.save_image((x + 1) / 2, SEL_DIR / "muestras_difusion_elegido.png", nrow=N_CLASSES)
if IN_COLAB:
    from IPython.display import Image as IPImage, display
    display(IPImage(filename=str(SEL_DIR / "muestras_difusion_elegido.png"), width=1000))

Output hidden; open in https://colab.research.google.com to view.

## Paso 9 · Registro

In [11]:
reg = {"fecha_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
       "tecnica": "DDPM condicional + CFG", "criterio": "min KD DINOv2-ViT-L/14 en validacion 128px",
       "n_generadas_por_eval": int(len(Y_SEL)), "ddim_steps": DDIM_STEPS, "min_step": MIN_STEP,
       "elegido": {"checkpoint": MEJOR["checkpoint"], "paso": int(MEJOR["paso"]), "guia": float(MEJOR["guia"]),
                   **{k: float(MEJOR[k]) for k in ["KD", "FD", "precision", "recall", "density", "coverage"]}},
       "seg_por_img": float(SEL_W["seg_por_img"].median()), "features": FEATURES,
       "versiones": {"python": sys.version.split()[0], "torch": torch.__version__, "diffusers": diffusers.__version__}}
(LOGS / "07a_seleccion_difusion.json").write_text(json.dumps(reg, indent=2, ensure_ascii=False))
print(json.dumps(reg["elegido"], indent=2, ensure_ascii=False))

{
  "checkpoint": "ema_0100000.pt",
  "paso": 100000,
  "guia": 2.0,
  "KD": 1.6184665795355606,
  "FD": 849.8160612978204,
  "precision": 0.3411764705882353,
  "recall": 0.6941176470588235,
  "density": 0.1776470588235294,
  "coverage": 0.34705882352941175
}


**Al terminar:** desconecta la GPU. Envía al asesor las tablas de los Pasos 6 y 7, el gráfico y la malla.